# 06. Quantization Method Selection Matrix and Head-to-Head Benchmark

- **Category:** 14_quantisation / 05_kv_attention_optim_and_benchmarks
- **Lineage & Citations:** Synthesis across Dettmers et al. (2022, 2023); Frantar et al. (2022); Lin et al. (2023); Wang et al. (2024); Xiao et al. (2023).
- **Purpose:** The Grand Capstone Synthesis of Category 14 (`14_quantisation`). Provides an exhaustive production selection matrix, Pareto frontier evaluation, and head-to-head empirical benchmarking across representative quantization paradigms.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [05. HAWQ Mixed Precision Bit Allocation and Numeric Debugging](05_hawq_mixed_precision_bit_allocation_and_numeric_debugging.ipynb) $\to$ `🔗 Category 14 Capstone:` Complete Curriculum Synthesis.

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT deploy uniform 2-bit scalar quantization for LLM serving (scalar 2-bit drops to negative SQNR; below 3 bits, always use multi-codebook vector/lattice quantization or native ternary pre-training); do NOT omit activation scaling when outliers exceed $6\sigma$ (use AWQ or QuaRot).

## Mathematical Foundations & Governing Equations

1. **Uniform Symmetric Affine Quantization (RTN)**:
   $$s_w = \frac{\max |W|}{2^{b-1} - 1}, \quad q = \text{clamp}\left(\left\lfloor \frac{W}{s_w} \right\rceil, -2^{b-1}+1, 2^{b-1}-1\right), \quad \hat{W} = q \cdot s_w$$
2. **Activation-Aware Weight Quantization (AWQ)**:
   $$s_{\text{awq}} = \sqrt{\mathbb{E}[|X|]}, \quad W' = W \cdot s_{\text{awq}}, \quad \hat{W} = \frac{Q(W')}{s_{\text{awq}}}$$
3. **BitNet b1.58 Native Ternary Quantization**:
   $$s = \frac{1}{d}\sum |W_i|, \quad \hat{W} = s \cdot \text{clamp}\left(\left\lfloor \frac{W}{s} \right\rceil, -1, 1\right)$$
4. **Information-Theoretic NormalFloat (NF4)**:
   $$\hat{W} = \arg\min_{c_k \in \mathcal{C}_{\text{NF4}}} \left| \frac{W}{\max |W|} - c_k \right| \cdot \max |W|$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/05_kv_attention_optim_and_benchmarks/06_quantization_method_selection_matrix_and_head_to_head_benchmark.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
NF4_LEVELS = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
    0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0
], dtype=torch.float32)

def rtn_quantize(w: torch.Tensor, bits: int) -> torch.Tensor:
    levels = float((2 ** (bits - 1)) - 1)
    scale = torch.amax(torch.abs(w), dim=-1, keepdim=True) / levels
    return torch.clamp(torch.round(w / (scale + 1e-8)), -levels, levels) * scale

def awq_quantize(w: torch.Tensor, x: torch.Tensor, bits: int = 4) -> torch.Tensor:
    levels = float((2 ** (bits - 1)) - 1)
    s_awq = torch.clamp(torch.mean(torch.abs(x), dim=0, keepdim=True) ** 0.5, min=1e-3)
    w_s = w * s_awq
    s_w = torch.amax(torch.abs(w_s), dim=-1, keepdim=True) / levels
    return (torch.clamp(torch.round(w_s / (s_w + 1e-8)), -levels, levels) * s_w) / s_awq

def bitnet_quantize(w: torch.Tensor) -> torch.Tensor:
    scale = torch.mean(torch.abs(w))
    return torch.clamp(torch.round(w / (scale + 1e-8)), -1.0, 1.0) * scale

def nf4_quantize(w: torch.Tensor) -> torch.Tensor:
    scale = torch.amax(torch.abs(w), dim=-1, keepdim=True)
    norm_w = w / (scale + 1e-8)
    diffs = torch.abs(norm_w.unsqueeze(-1) - NF4_LEVELS)
    return NF4_LEVELS[torch.argmin(diffs, dim=-1)] * scale

def compute_sqnr(sig: torch.Tensor, noise: torch.Tensor) -> float:
    return float(10.0 * torch.log10(torch.mean(sig ** 2) / (torch.mean(noise ** 2) + 1e-12)))

In [ ]:
# Parity Verification against golden checkpoint
y_anchor = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

w_t, x_t = weights["w"], weights["x"]

w_rtn8 = rtn_quantize(w_t, 8)
assert float(torch.max(torch.abs(w_rtn8 - weights["w_rtn8"]))) < 1e-5

w_rtn4 = rtn_quantize(w_t, 4)
assert float(torch.max(torch.abs(w_rtn4 - weights["w_rtn4"]))) < 1e-5

w_awq4 = awq_quantize(w_t, x_t, 4)
assert float(torch.max(torch.abs(w_awq4 - weights["w_awq4"]))) < 1e-5

w_bitnet = bitnet_quantize(w_t)
assert float(torch.max(torch.abs(w_bitnet - weights["w_bitnet"]))) < 1e-5

w_nf4 = nf4_quantize(w_t)
assert float(torch.max(torch.abs(w_nf4 - weights["w_nf4"]))) < 1e-5

methods = [w_rtn8, w_rtn4, w_awq4, w_bitnet, w_nf4]
y_true = x_t @ w_t

for i, w_q in enumerate(methods):
    w_sqnr = compute_sqnr(w_t, w_t - w_q)
    act_sqnr = compute_sqnr(y_true, y_true - (x_t @ w_q))
    assert abs(w_sqnr - weights["weight_sqnrs"][i].item()) < 1e-3
    assert abs(act_sqnr - weights["act_sqnrs"][i].item()) < 1e-3
print("All 5 representative quantization algorithms verified against golden checkpoint!")

In [ ]:
# Head-to-Head Benchmark Table Generation
names = ["RTN INT8", "RTN INT4", "AWQ INT4", "BitNet b1.58", "NF4 4-bit"]
bits = [8.0, 4.0, 4.0, 1.58, 4.0]

print(f"{'Method':14s} | {'Bits':5s} | {'Comp Ratio':10s} | {'Weight SQNR':12s} | {'Output SQNR':12s}")
print("-" * 65)
for i, name in enumerate(names):
    b = bits[i]
    comp = 32.0 / b
    w_s = weights["weight_sqnrs"][i].item()
    a_s = weights["act_sqnrs"][i].item()
    print(f"{name:14s} | {b:5.2f} | {comp:9.2f}x | {w_s:10.2f} dB | {a_s:10.2f} dB")

assert weights["act_sqnrs"][0] > 40.0, "INT8 must exceed 40 dB SQNR!"
assert weights["act_sqnrs"][4] > weights["act_sqnrs"][1], "NF4 must outperform uniform INT4!"

In [ ]:
# Latency Microbenchmark: Multi-Format Quantization Dispatch (512x512 parameters)
w_bench = torch.randn(512, 512) * 0.1
x_bench = torch.randn(32, 512)

timings = {}
for fn_name, fn in [("RTN8", lambda: rtn_quantize(w_bench, 8)),
                    ("RTN4", lambda: rtn_quantize(w_bench, 4)),
                    ("AWQ4", lambda: awq_quantize(w_bench, x_bench, 4)),
                    ("BitNet", lambda: bitnet_quantize(w_bench)),
                    ("NF4", lambda: nf4_quantize(w_bench))]:
    t0 = time.perf_counter()
    for _ in range(50): _ = fn()
    timings[fn_name] = (time.perf_counter() - t0) / 50 * 1e3
    print(f"{fn_name:8s} Execution Latency: {timings[fn_name]:.3f} ms")

assert timings["RTN8"] < 20.0, "Quantization routines must complete in <20 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Information-Theoretic Quantiles Beat Uniform**: NF4 matches normal weight distributions to outperform uniform INT4 by $+1.52\text{ dB}$ output fidelity without calibration.
2. **Activation Protection Prevents Divergence**: In networks with kurtosis outliers, AWQ preserves critical activation channels to maintain accuracy where unscaled low-bit fails.
3. **Ternary Computing Eliminates Multiplications**: BitNet b1.58 delivers $20.25\times$ weight compression and replaces floating-point multipliers with low-energy integer additions.

---
### Synchronized Track Navigation
- [Companion Synthesis Guide](../../../markdown/14_quantisation/05_kv_attention_optim_and_benchmarks/06_quantization_method_selection_matrix_and_head_to_head_benchmark.md)
- [PyTorch Reference](../../../pytorch2/14_quantisation/05_kv_attention_optim_and_benchmarks/06_quantization_method_selection_matrix_and_head_to_head_benchmark.py)
- [JAX Reference](../../../jax/14_quantisation/05_kv_attention_optim_and_benchmarks/06_quantization_method_selection_matrix_and_head_to_head_benchmark.py)
- [Candle Reference](../../../candle/14_quantisation/05_kv_attention_optim_and_benchmarks/06_quantization_method_selection_matrix_and_head_to_head_benchmark.rs)
- [Kernels Reference](../../../kernels/14_quantisation/05_kv_attention_optim_and_benchmarks/06_quantization_method_selection_matrix_and_head_to_head_benchmark.rs)
- `🔗 Upstream Prerequisites:` [05. HAWQ Mixed Precision Bit Allocation and Numeric Debugging](05_hawq_mixed_precision_bit_allocation_and_numeric_debugging.ipynb)
- **Category 14 (Quantisation) 100% COMPLETE!**